# 5 Fold Cross validation with yolo 11m.

In [1]:
# YOLOv11m Segmentation Training with 5-Fold Cross-Validation
# This script sets up and runs a 5-fold cross-validation training for a YOLO model.
# It handles the data preparation, training for each fold, and aggregation of results.

# ## 1. Setup
# Import libraries and define paths.
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO
import torch

# Define paths
ROOT_DIR = Path('../') # Running from the root of the project
DATA_DIR = ROOT_DIR / 'data' / 'processed'
CV_DATA_DIR = ROOT_DIR / 'cv_data_yolo11m' # New directory for this CV
MODEL_CONFIG_PATH = ROOT_DIR / 'configurations' / 'model_data-seg.yaml'
# Using the best.pt from the user's previous yolo11m training run
PRETRAINED_MODEL_PATH = ROOT_DIR / 'notebooks' / 'runs' / 'segment' / 'train_Yolo11m_canopy_adamW_' / 'weights' / 'best.pt'


# ## 2. Data Preparation for Cross-Validation
# Combine the existing training and validation sets and then split them into 5 folds.
print("--- Preparing Data for Cross-Validation ---")

# Combine all images and labels
all_images = sorted(list(DATA_DIR.glob('images/train/*.tif'))) + sorted(list(DATA_DIR.glob('images/val/*.tif')))
all_labels = sorted(list(DATA_DIR.glob('labels/train/*.txt'))) + sorted(list(DATA_DIR.glob('labels/val/*.txt')))

# Ensure images and labels correspond
assert len(all_images) == len(all_labels), "Mismatch between number of images and labels"
for img, lbl in zip(all_images, all_labels):
    assert img.stem == lbl.stem, f"Mismatch between {img.name} and {lbl.name}"

print(f"Total images for CV: {len(all_images)}")

# Create the main CV directory
if CV_DATA_DIR.exists():
    shutil.rmtree(CV_DATA_DIR)
CV_DATA_DIR.mkdir(exist_ok=True)

# Setup KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_images = np.array(all_images)

# Get class names from original data config
with open(MODEL_CONFIG_PATH, 'r') as f:
    model_config = yaml.safe_load(f)
class_names = model_config['names']

for i, (train_index, val_index) in enumerate(kf.split(all_images)):
    fold_dir = CV_DATA_DIR / f'fold_{i}'
    
    # Create directories for the fold
    (fold_dir / 'images' / 'train').mkdir(parents=True, exist_ok=True)
    (fold_dir / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (fold_dir / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
    (fold_dir / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
    
    # Copy training files
    for idx in train_index:
        img_path = all_images[idx]
        lbl_path = Path(str(img_path).replace('images', 'labels').replace('.tif', '.txt'))
        shutil.copy(img_path, fold_dir / 'images' / 'train' / img_path.name)
        shutil.copy(lbl_path, fold_dir / 'labels' / 'train' / lbl_path.name)
        
    # Copy validation files
    for idx in val_index:
        img_path = all_images[idx]
        lbl_path = Path(str(img_path).replace('images', 'labels').replace('.tif', '.txt'))
        shutil.copy(img_path, fold_dir / 'images' / 'val' / img_path.name)
        shutil.copy(lbl_path, fold_dir / 'labels' / 'val' / lbl_path.name)
        
    # Create YAML file for the fold
    fold_yaml_path = fold_dir / f'fold_{i}_data.yaml'
    fold_data_config = {
        'path': str(fold_dir.resolve()),
        'train': str((fold_dir / 'images' / 'train').resolve()),
        'val': str((fold_dir / 'images' / 'val').resolve()),
        'names': class_names
    }
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_data_config, f)
        
    print(f"Fold {i} created.")

print("--- Data Preparation Complete ---")

# ## 3. Cross-Validation Training
# Loop through each fold and train the model.
for i in range(5):
    print(f"\n--- Training Fold {i} ---")
    
    # Load pretrained model
    model = YOLO(PRETRAINED_MODEL_PATH)
    
    # Get data config for the fold
    fold_yaml_path = CV_DATA_DIR / f'fold_{i}' / f'fold_{i}_data.yaml'
    
    # Train the model
    model.train(
        data=str(fold_yaml_path.resolve()),
        epochs=25, # Using a smaller number of epochs for demonstration
        imgsz=640,
        batch=2,
        project='YOLOv11m_CV',
        name=f'fold_{i}',
        exist_ok=True # Allows re-running the script
    )
    
    # Clear memory
    del model
    torch.cuda.empty_cache()

print("--- Cross-Validation Training Complete ---")

# ## 4. Results Aggregation
# Combine the results from all folds and calculate the mean and standard deviation of the key metrics.
print("\n--- Aggregating Results ---")
results_dir = ROOT_DIR / 'YOLOv11m_CV'
all_results = []

for i in range(5):
    fold_results_path = results_dir / f'fold_{i}' / 'results.csv'
    if fold_results_path.exists():
        df = pd.read_csv(fold_results_path)
        df['fold'] = i
        all_results.append(df)

if all_results:
    cv_results_df = pd.concat(all_results)
    
    # Get the metrics from the last epoch of each fold
    last_epoch_results = cv_results_df.groupby('fold').last().reset_index()
    
    print("\n--- Cross-Validation Results Summary (Last Epoch of Each Fold) ---")
    print(last_epoch_results[['fold', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']])
    
    print("\n--- Mean and Std Dev of Metrics ---")
    mean_metrics = last_epoch_results[['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']].mean()
    std_metrics = last_epoch_results[['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']].std()
    
    summary_df = pd.DataFrame({'mean': mean_metrics, 'std_dev': std_metrics})
    print(summary_df)
else:
    print("No results found. Please ensure the training completed successfully.")

print("\n--- Script Finished ---")


--- Preparing Data for Cross-Validation ---
Total images for CV: 150
Fold 0 created.
Fold 1 created.
Fold 2 created.
Fold 3 created.
Fold 4 created.
--- Data Preparation Complete ---

--- Training Fold 0 ---
New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.185  Python-3.11.14 torch-2.0.1+cu117 CUDA:0 (NVIDIA RTX A2000 12GB, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\fold_0_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv

train: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\train... 120 images


train: New cache created: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\train.cache
val: Fast image access  (ping: 0.20.1 ms, read: 153.779.7 MB/s, size: 3072.3 KB)


val: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\val... 30 images, 0 b

val: New cache created: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\val.cache


...

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         30       9379      0.507      0.328      0.361      0.194      0.469      0.301       0.33      0.148
       individual_tree         30       8299      0.613      0.374      0.466      0.257      0.516      0.312      0.387      0.179
        group_of_trees         25       1080      0.401      0.282      0.257      0.131      0.422      0.291      0.272      0.118
Speed: 6.1ms preprocess, 99.1ms inference, 0.0ms loss, 15.1ms postprocess per image
Results saved to YOLOv11m_CV\fold_4
--- Cross-Validation Training Complete ---

--- Aggregating Results ---
No results found. Please ensure the training completed successfully.

--- Script Finished ---


In [4]:
import torch
torch.cuda.empty_cache()


In [4]:
# ## 4. Results Aggregation (with debugging)
print("\n--- Aggregating Results (Debug Mode) ---")

results_dir = ROOT_DIR / 'notebooks' / 'YOLOv11m_CV'
print(f"[DEBUG] Looking for results in: {results_dir.resolve()}")

all_results = []

for i in range(5):
    fold_dir = results_dir / f'fold_{i}'
    fold_results_path = fold_dir / 'results.csv'
    
    print(f"\n[DEBUG] Checking fold {i}:")
    print(f"[DEBUG] Expected path: {fold_results_path.resolve()}")
    print(f"[DEBUG] Exists? {fold_results_path.exists()}")
    
    if fold_results_path.exists():
        try:
            df = pd.read_csv(fold_results_path)
            print(f"[DEBUG] Loaded results.csv for fold {i}")
            print(f"[DEBUG] Columns: {df.columns.tolist()}")
            df['fold'] = i
            all_results.append(df)
        except Exception as e:
            print(f"[ERROR] Failed to read results.csv for fold {i}: {e}")

if not all_results:
    print("\n[ERROR] No results.csv files were loaded.")
    print("Check folder paths printed above — they must match where YOLO actually saved outputs.")
    print("\n--- Script Finished ---")
else:
    cv_results_df = pd.concat(all_results)
    
    # Get the metrics from the last epoch of each fold
    last_epoch_results = cv_results_df.groupby('fold').last().reset_index()

    print("\n--- Cross-Validation Results Summary (Last Epoch of Each Fold) ---")
    try:
        print(last_epoch_results[['fold',
                                  'metrics/mAP50(B)',
                                  'metrics/mAP50-95(B)',
                                  'metrics/mAP50(M)',
                                  'metrics/mAP50-95(M)']])
    except KeyError:
        print("\n[ERROR] Some metric columns were not found.")
        print("Available columns:", last_epoch_results.columns.tolist())

    print("\n--- Mean and Std Dev of Metrics ---")
    try:
        mean_metrics = last_epoch_results[['metrics/mAP50(B)',
                                           'metrics/mAP50-95(B)',
                                           'metrics/mAP50(M)',
                                           'metrics/mAP50-95(M)']].mean()

        std_metrics = last_epoch_results[['metrics/mAP50(B)',
                                          'metrics/mAP50-95(B)',
                                          'metrics/mAP50(M)',
                                          'metrics/mAP50-95(M)']].std()

        summary_df = pd.DataFrame({'mean': mean_metrics, 'std_dev': std_metrics})
        print(summary_df)
    except KeyError:
        print("\n[ERROR] Could not compute mean/std because metric columns are missing.")

    print("\n--- Script Finished ---")



--- Aggregating Results (Debug Mode) ---
[DEBUG] Looking for results in: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\YOLOv11m_CV

[DEBUG] Checking fold 0:
[DEBUG] Expected path: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\YOLOv11m_CV\fold_0\results.csv
[DEBUG] Exists? True
[DEBUG] Loaded results.csv for fold 0
[DEBUG] Columns: ['epoch', 'time', 'train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(M)', 'metrics/recall(M)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)', 'val/box_loss', 'val/seg_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

[DEBUG] Checking fold 1:
[DEBUG] Expected path: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\YOLOv11m_CV\fold_1\results.csv
[DEBUG] Exists? True
[DEBUG] Loaded results.csv for fold 1
[DEBUG] Columns: ['epoch', 'time', 'train/box_loss', 'train/seg_los

In [5]:
# YOLOv11s Segmentation Training with 5-Fold Cross-Validation
# Updated per suggestions: 150 epochs, early stopping, stronger augmentations, accumulation, YOLOv11s model.

# ## 1. Setup
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO
import torch

ROOT_DIR = Path('../')
DATA_DIR = ROOT_DIR / 'data' / 'processed'
CV_DATA_DIR = ROOT_DIR / 'cv_data_yolo11s'
MODEL_CONFIG_PATH = ROOT_DIR / 'configurations' / 'model_data-seg.yaml'

# Switch to YOLOv11s segmentation model (improves generalization on small datasets)
PRETRAINED_MODEL_PATH = "yolo11s-seg.pt"  # Ultralytics auto-downloads if not found

print("--- Preparing Data for Cross-Validation ---")

all_images = sorted(list(DATA_DIR.glob('images/train/*.tif'))) + \
             sorted(list(DATA_DIR.glob('images/val/*.tif')))

all_labels = sorted(list(DATA_DIR.glob('labels/train/*.txt'))) + \
             sorted(list(DATA_DIR.glob('labels/val/*.txt')))

assert len(all_images) == len(all_labels), "Mismatch between number of images and labels"
for img, lbl in zip(all_images, all_labels):
    assert img.stem == lbl.stem, f"Mismatch between {img.name} and {lbl.name}"

print(f"Total images for CV: {len(all_images)}")

if CV_DATA_DIR.exists():
    shutil.rmtree(CV_DATA_DIR)
CV_DATA_DIR.mkdir(exist_ok=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_images = np.array(all_images)

with open(MODEL_CONFIG_PATH, 'r') as f:
    model_config = yaml.safe_load(f)
class_names = model_config['names']

for i, (train_index, val_index) in enumerate(kf.split(all_images)):
    fold_dir = CV_DATA_DIR / f'fold_{i}'
    (fold_dir / 'images' / 'train').mkdir(parents=True, exist_ok=True)
    (fold_dir / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (fold_dir / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
    (fold_dir / 'labels' / 'val').mkdir(parents=True, exist_ok=True)

    for idx in train_index:
        img_path = all_images[idx]
        lbl_path = Path(str(img_path).replace('images', 'labels').replace('.tif', '.txt'))
        shutil.copy(img_path, fold_dir / 'images' / 'train' / img_path.name)
        shutil.copy(lbl_path, fold_dir / 'labels' / 'train' / lbl_path.name)

    for idx in val_index:
        img_path = all_images[idx]
        lbl_path = Path(str(img_path).replace('images', 'labels').replace('.tif', '.txt'))
        shutil.copy(img_path, fold_dir / 'images' / 'val' / img_path.name)
        shutil.copy(lbl_path, fold_dir / 'labels' / 'val' / lbl_path.name)

    fold_yaml_path = fold_dir / f'fold_{i}_data.yaml'
    fold_data_config = {
        'path': str(fold_dir.resolve()),
        'train': str((fold_dir / 'images' / 'train').resolve()),
        'val': str((fold_dir / 'images' / 'val').resolve()),
        'names': class_names,

        # 🔥 Data Augmentation — moderate, safe, effective for small datasets
        'augmentation': {
            'mosaic': 1.0,
            'mixup': 0.2,
            'copy_paste': 0.3,
            'hsv_h': 0.015,
            'hsv_s': 0.7,
            'hsv_v': 0.4,
            'fliplr': 0.5,
            'flipud': 0.0,
            'perspective': 0.01
        }
    }

    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_data_config, f)

    print(f"Fold {i} created.")

print("--- Data Preparation Complete ---")


# ## 3. Cross-Validation Training
for i in range(5):
    print(f"\n--- Training Fold {i} ---")

    model = YOLO(PRETRAINED_MODEL_PATH)
    fold_yaml_path = CV_DATA_DIR / f'fold_{i}' / f'fold_{i}_data.yaml'

    model.train(
        data=str(fold_yaml_path.resolve()),
        epochs=150,               # ⬆️ increased epochs (suggestion #1)
        imgsz=640,
        batch=-1,           
        patience=20,             # ⬆️ early stopping
        optimizer="AdamW",
        lr0=0.0005,
        project='YOLOv11s_CV',
        name=f'fold_{i}',
        exist_ok=True
    )

    del model
    torch.cuda.empty_cache()

print("--- Cross-Validation Training Complete ---")


# ## 4. Results Aggregation
print("\n--- Aggregating Results ---")
results_dir = ROOT_DIR / 'notebooks' / 'runs' / 'segment' / 'YOLOv11s_CV'
all_results = []

for i in range(5):
    fold_results_path = results_dir / f'fold_{i}' / 'results.csv'
    print(f"[DEBUG] Looking for: {fold_results_path}  Exists: {fold_results_path.exists()}")

    if fold_results_path.exists():
        df = pd.read_csv(fold_results_path)
        df['fold'] = i
        all_results.append(df)

if all_results:
    cv_results_df = pd.concat(all_results)
    last_epoch_results = cv_results_df.groupby('fold').last().reset_index()

    print("\n--- Cross-Validation Results Summary (Last Epoch) ---")
    print(last_epoch_results[['fold', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
                              'metrics/mAP50(M)', 'metrics/mAP50-95(M)']])

    print("\n--- Mean and Std Dev of Metrics ---")
    summary_df = pd.DataFrame({
        'mean': last_epoch_results[['metrics/mAP50(B)',
                                    'metrics/mAP50-95(B)',
                                    'metrics/mAP50(M)',
                                    'metrics/mAP50-95(M)']].mean(),
        'std_dev': last_epoch_results[['metrics/mAP50(B)',
                                       'metrics/mAP50-95(B)',
                                       'metrics/mAP50(M)',
                                       'metrics/mAP50-95(M)']].std()
    })
    print(summary_df)
else:
    print("No results found. Please ensure the training completed successfully.")

print("\n--- Script Finished ---")


--- Preparing Data for Cross-Validation ---
Total images for CV: 150
ERROR! Session/line number was not unique in database. History logging moved to new session 124
Fold 0 created.
Fold 1 created.
Fold 2 created.
Fold 3 created.
Fold 4 created.
--- Data Preparation Complete ---

--- Training Fold 0 ---
New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.185  Python-3.11.14 torch-2.0.1+cu117 CUDA:0 (NVIDIA RTX A2000 12GB, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11s\fold_0\fold_0_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, f

train: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11s\fold_0\labels\train... 120 images


train: New cache created: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11s\fold_0\labels\train.cache
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (NVIDIA RTX A2000 12GB) 11.99G total, 1.59G reserved, 0.58G allocated, 9.82G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    10083062       33.08         1.441         94.12           nan        (1, 3, 640, 640)                    list
    10083062       66.16         2.554         73.88           nan        (2, 3, 640, 640)                    list
    10083062       132.3         4.379         40.47           nan        (4, 3, 640, 640)                    list
    10083062       264.6         7.636         65.69           nan        (8, 3, 640, 640)                    list
    10083062       529.2        14.875         135.8           nan       (16, 3, 640, 640)                    list


train: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11s\fold_0\labels\train.cache... 120 


val: Fast image access  (ping: 0.10.0 ms, read: 199.338.7 MB/s, size: 3072.3 KB)


...

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         30       9379       0.45      0.283      0.326      0.171      0.421       0.28      0.312      0.138
       individual_tree         30       8299      0.531      0.333      0.405      0.224      0.474      0.312      0.371      0.168
        group_of_trees         25       1080       0.37      0.234      0.247      0.118      0.368      0.248      0.253      0.107
Speed: 1.4ms preprocess, 11.6ms inference, 0.0ms loss, 32.5ms postprocess per image
Results saved to YOLOv11s_CV\fold_4
--- Cross-Validation Training Complete ---

--- Aggregating Results ---
[DEBUG] Looking for: ..\notebooks\runs\segment\YOLOv11s_CV\fold_0\results.csv  Exists: False
[DEBUG] Looking for: ..\notebooks\runs\segment\YOLOv11s_CV\fold_1\results.csv  Exists: False
[DEBUG] Looking for: ..\notebooks\runs\segment\YOLOv11s_CV\fold_2\results.csv  Exists: False
[DEBUG] Looking for: ..\notebooks\runs\segment\YOLOv11s_CV\fold_3\results.csv  Exists: False
[DEBUG] Looking for: ..\notebooks\run

In [7]:
# ## 4. Results Aggregation (with debugging)
print("\n--- Aggregating Results (Debug Mode) ---")

results_dir = ROOT_DIR / 'notebooks' / 'YOLOv11s_CV'
print(f"[DEBUG] Looking for results in: {results_dir.resolve()}")

all_results = []

for i in range(5):
    fold_dir = results_dir / f'fold_{i}'
    fold_results_path = fold_dir / 'results.csv'
    
    print(f"\n[DEBUG] Checking fold {i}:")
    print(f"[DEBUG] Expected path: {fold_results_path.resolve()}")
    print(f"[DEBUG] Exists? {fold_results_path.exists()}")
    
    if fold_results_path.exists():
        try:
            df = pd.read_csv(fold_results_path)
            print(f"[DEBUG] Loaded results.csv for fold {i}")
            print(f"[DEBUG] Columns: {df.columns.tolist()}")
            df['fold'] = i
            all_results.append(df)
        except Exception as e:
            print(f"[ERROR] Failed to read results.csv for fold {i}: {e}")

if not all_results:
    print("\n[ERROR] No results.csv files were loaded.")
    print("Check folder paths printed above — they must match where YOLO actually saved outputs.")
    print("\n--- Script Finished ---")
else:
    cv_results_df = pd.concat(all_results)
    
    # Get the metrics from the last epoch of each fold
    last_epoch_results = cv_results_df.groupby('fold').last().reset_index()

    print("\n--- Cross-Validation Results Summary (Last Epoch of Each Fold) ---")
    try:
        print(last_epoch_results[['fold',
                                  'metrics/mAP50(B)',
                                  'metrics/mAP50-95(B)',
                                  'metrics/mAP50(M)',
                                  'metrics/mAP50-95(M)']])
    except KeyError:
        print("\n[ERROR] Some metric columns were not found.")
        print("Available columns:", last_epoch_results.columns.tolist())

    print("\n--- Mean and Std Dev of Metrics ---")
    try:
        mean_metrics = last_epoch_results[['metrics/mAP50(B)',
                                           'metrics/mAP50-95(B)',
                                           'metrics/mAP50(M)',
                                           'metrics/mAP50-95(M)']].mean()

        std_metrics = last_epoch_results[['metrics/mAP50(B)',
                                          'metrics/mAP50-95(B)',
                                          'metrics/mAP50(M)',
                                          'metrics/mAP50-95(M)']].std()

        summary_df = pd.DataFrame({'mean': mean_metrics, 'std_dev': std_metrics})
        print(summary_df)
    except KeyError:
        print("\n[ERROR] Could not compute mean/std because metric columns are missing.")

    print("\n--- Script Finished ---")



--- Aggregating Results (Debug Mode) ---
[DEBUG] Looking for results in: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\YOLOv11s_CV

[DEBUG] Checking fold 0:
[DEBUG] Expected path: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\YOLOv11s_CV\fold_0\results.csv
[DEBUG] Exists? True
[DEBUG] Loaded results.csv for fold 0
[DEBUG] Columns: ['epoch', 'time', 'train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(M)', 'metrics/recall(M)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)', 'val/box_loss', 'val/seg_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

[DEBUG] Checking fold 1:
[DEBUG] Expected path: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\YOLOv11s_CV\fold_1\results.csv
[DEBUG] Exists? True
[DEBUG] Loaded results.csv for fold 1
[DEBUG] Columns: ['epoch', 'time', 'train/box_loss', 'train/seg_los

In [8]:
# YOLOv11m Segmentation Training with 5-Fold Cross-Validation
# 150 epochs, early stopping, strong augmentations, full-size images, unique results folder

import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO
import torch
from datetime import datetime

# -----------------------
# Paths
# -----------------------
ROOT_DIR = Path('../')
DATA_DIR = ROOT_DIR / 'data' / 'processed'
CV_DATA_DIR = ROOT_DIR / 'cv_data_yolo11m'
MODEL_CONFIG_PATH = ROOT_DIR / 'configurations' / 'model_data-seg.yaml'
PRETRAINED_MODEL_PATH = ROOT_DIR / 'notebooks' / 'runs' / 'segment' / 'train_Yolo11m_canopy_adamW_' / 'weights' / 'best.pt'

# Unique results folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_FOLDER = ROOT_DIR / f'YOLOv11m_CV_{timestamp}'
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

print("--- Preparing Data for Cross-Validation ---")

# -----------------------
# Combine all images and labels
# -----------------------
all_images = sorted(list(DATA_DIR.glob('images/train/*.tif'))) + \
             sorted(list(DATA_DIR.glob('images/val/*.tif')))
all_labels = sorted(list(DATA_DIR.glob('labels/train/*.txt'))) + \
             sorted(list(DATA_DIR.glob('labels/val/*.txt')))

assert len(all_images) == len(all_labels), "Mismatch between images and labels"
for img, lbl in zip(all_images, all_labels):
    assert img.stem == lbl.stem, f"Mismatch {img.name} vs {lbl.name}"

print(f"Total images: {len(all_images)}")

# -----------------------
# Prepare CV directories
# -----------------------
if CV_DATA_DIR.exists():
    shutil.rmtree(CV_DATA_DIR)
CV_DATA_DIR.mkdir(exist_ok=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_images = np.array(all_images)

with open(MODEL_CONFIG_PATH, 'r') as f:
    model_config = yaml.safe_load(f)
class_names = model_config['names']

# -----------------------
# Prepare folds
# -----------------------
for i, (train_idx, val_idx) in enumerate(kf.split(all_images)):
    fold_dir = CV_DATA_DIR / f"fold_{i}"
    img_train_dir = fold_dir / "images/train"
    img_val_dir = fold_dir / "images/val"
    mask_train_dir = fold_dir / "labels/train"
    mask_val_dir = fold_dir / "labels/val"

    img_train_dir.mkdir(parents=True, exist_ok=True)
    img_val_dir.mkdir(parents=True, exist_ok=True)
    mask_train_dir.mkdir(parents=True, exist_ok=True)
    mask_val_dir.mkdir(parents=True, exist_ok=True)

    # Copy images/masks
    for idx in train_idx:
        img_path = all_images[idx]
        mask_path = Path(str(img_path).replace("images", "labels").replace(".tif", ".txt"))
        shutil.copy(img_path, img_train_dir / img_path.name)
        shutil.copy(mask_path, mask_train_dir / mask_path.name)

    for idx in val_idx:
        img_path = all_images[idx]
        mask_path = Path(str(img_path).replace("images", "labels").replace(".tif", ".txt"))
        shutil.copy(img_path, img_val_dir / img_path.name)
        shutil.copy(mask_path, mask_val_dir / mask_path.name)

    # Write fold YAML
    fold_yaml_path = fold_dir / f"fold_{i}_data.yaml"
    fold_data_config = {
        'path': str(fold_dir.resolve()),
        'train': str(img_train_dir.resolve()),
        'val': str(img_val_dir.resolve()),
        'names': class_names,
        'augmentation': {
            'mosaic': 1.0,
            'mixup': 0.2,
            'copy_paste': 0.3,
            'hsv_h': 0.015,
            'hsv_s': 0.7,
            'hsv_v': 0.4,
            'fliplr': 0.5,
            'flipud': 0.0,
            'perspective': 0.01
        }
    }
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_data_config, f)

    print(f"Fold {i} prepared.")

print("--- Data preparation complete ---")

# -----------------------
# Training per fold
# -----------------------
for i in range(5):
    print(f"\n--- Training Fold {i} ---")
    model = YOLO(PRETRAINED_MODEL_PATH)
    fold_yaml_path = CV_DATA_DIR / f"fold_{i}" / f"fold_{i}_data.yaml"

    model.train(
        data=str(fold_yaml_path.resolve()),
        epochs=150,
        imgsz=640,
        batch=-1,           # auto-adjust batch size & gradient accumulation
        patience=20,        # early stopping
        optimizer="AdamW",
        lr0=0.0005,
        project=str(RESULTS_FOLDER),
        name=f'fold_{i}',
        exist_ok=True
    )

    del model
    torch.cuda.empty_cache()

print("--- Cross-validation training complete ---")

# -----------------------
# Results aggregation
# -----------------------
print("\n--- Aggregating Results ---")
all_results = []

for i in range(5):
    fold_results_path = RESULTS_FOLDER / f'fold_{i}' / 'results.csv'
    print(f"[DEBUG] Looking for: {fold_results_path}  Exists: {fold_results_path.exists()}")
    if fold_results_path.exists():
        df = pd.read_csv(fold_results_path)
        df['fold'] = i
        all_results.append(df)

if all_results:
    cv_results_df = pd.concat(all_results)
    last_epoch_results = cv_results_df.groupby('fold').last().reset_index()
    print("\n--- Cross-Validation Results Summary (Last Epoch) ---")
    print(last_epoch_results[['fold', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
                              'metrics/mAP50(M)', 'metrics/mAP50-95(M)']])

    summary_df = pd.DataFrame({
        'mean': last_epoch_results[['metrics/mAP50(B)',
                                    'metrics/mAP50-95(B)',
                                    'metrics/mAP50(M)',
                                    'metrics/mAP50-95(M)']].mean(),
        'std_dev': last_epoch_results[['metrics/mAP50(B)',
                                       'metrics/mAP50-95(B)',
                                       'metrics/mAP50(M)',
                                       'metrics/mAP50-95(M)']].std()
    })
    print("\n--- Mean and Std Dev ---")
    print(summary_df)
else:
    print("No results found. Check paths and training outputs.")

print("\n--- Script Finished ---")


--- Preparing Data for Cross-Validation ---
Total images: 150
Fold 0 prepared.
Fold 1 prepared.
Fold 2 prepared.
Fold 3 prepared.
Fold 4 prepared.
--- Data preparation complete ---

--- Training Fold 0 ---
New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.185  Python-3.11.14 torch-2.0.1+cu117 CUDA:0 (NVIDIA RTX A2000 12GB, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\fold_0_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv

train: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\train... 120 images


train: New cache created: C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\train.cache
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (NVIDIA RTX A2000 12GB) 11.99G total, 0.66G reserved, 0.40G allocated, 10.94G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    22360758       113.5         2.439         154.6           nan        (1, 3, 640, 640)                    list
    22360758         227         4.782         49.12           nan        (2, 3, 640, 640)                    list
    22360758       454.1         8.066         46.94           nan        (4, 3, 640, 640)                    list
    22360758       908.1        14.655         124.7           nan        (8, 3, 640, 640)                    list
CUDA out of memory. Tried to allocate 50.00 MiB (GPU 0; 11.99 GiB total capacity; 10.51 GiB already allocated; 0 b

train: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\cv_data_yolo11m\fold_0\labels\train.cache... 120 


val: Fast image access  (ping: 0.10.0 ms, read: 168.322.0 MB/s, size: 3072.3 KB)


...

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         30       9379      0.503      0.331      0.368      0.198      0.463      0.313      0.344      0.159
       individual_tree         30       8299      0.605      0.369      0.453      0.251      0.531       0.33        0.4      0.186
        group_of_trees         25       1080        0.4      0.293      0.284      0.146      0.395      0.295      0.289      0.131
Speed: 3.7ms preprocess, 114.0ms inference, 0.0ms loss, 25.4ms postprocess per image
Results saved to ..\YOLOv11m_CV_20251207_181700\fold_4
--- Cross-validation training complete ---

--- Aggregating Results ---
[DEBUG] Looking for: ..\YOLOv11m_CV_20251207_181700\fold_0\results.csv  Exists: True
[DEBUG] Looking for: ..\YOLOv11m_CV_20251207_181700\fold_1\results.csv  Exists: True
[DEBUG] Looking for: ..\YOLOv11m_CV_20251207_181700\fold_2\results.csv  Exists: True
[DEBUG] Looking for: ..\YOLOv11m_CV_20251207_181700\fold_3\results.csv  Exists: True
[DEBUG] Looking for: ..\YOLOv11m_CV_20251207_1817